# Funções — demonstração exercícios e quando usar

Use funções para tarefas repetidas, regras de negócio e partes de uma solução que têm uma responsabilidade clara. Parâmetros recebem dados; return devolve um resultado reutilizável. print apenas mostra algo. Evite transformar cada linha simples em uma função.

Execute as demonstrações; depois tente os exercícios antes do gabarito. Escreva dados disponíveis, resultado esperado e passos em português. Requisitos: Python 3.9 ou superior, pandas e IPython. Para fusos, o ambiente precisa de dados IANA; se houver ZoneInfoNotFoundError, instale tzdata no kernel. Não é necessário baixar datasets.

In [3]:
import pandas as pd
from IPython.display import display

## Demo Python puro
O desconto é um percentual de 0 a 100. O total é preço vezes quantidade menos desconto. A função valida entradas antes de calcular; para esta aula usamos números comuns e float. Sistemas financeiros precisam de regras de arredondamento e representação decimal próprias.

In [2]:
def total_compra(preco, quantidade, desconto=0):
    if preco < 0 or quantidade < 0:
        raise ValueError("Preço e quantidade precisam ser não negativos.")
    if not 0 <= desconto <= 100:
        raise ValueError("Desconto precisa estar entre 0 e 100.")
    return preco * quantidade * (1 - desconto/100)

total = total_compra(50, 2, 10)
print(total)
assert total == 90

90.0


## Demo Pandas
Operações vetorizadas são úteis para cálculos entre colunas. apply é uma opção para uma regra personalizada sobre cada valor; funções não são obrigatórias para contas simples.

In [3]:
vendas = pd.DataFrame({"produto":["A","B","C"],"preco":[50,100,20],"quantidade":[2,3,1],"desconto":[10,0,5]})
vendas["total"] = vendas["preco"] * vendas["quantidade"] * (1-vendas["desconto"]/100)
display(vendas)

,produto,preco,quantidade,desconto,total
0,A,50,2,10,90.0
1,B,100,3,0,300.0
2,C,20,1,5,19.0


## Exercícios

### 1 Classificar compras
Crie classificar_compra(total): até menos de 100 Pequena; de 100 até menos de 300 Média; 300 ou mais Grande. Rejeite negativos. Teste os limites.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [4]:
# Sua solução aqui.

### 2 Reutilizar em lista e tabela
Classifique os totais [90, 300, 19] usando for. Depois aplique a mesma função à coluna total de vendas.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [5]:
# Sua solução aqui.

### 3 Compor funções
Crie resumir_compra(preco, quantidade, desconto=0) que retorna um dicionário com total e classe, reutilizando as duas funções anteriores. Teste desconto inválido e capture o erro.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [6]:
# Sua solução aqui.

## Gabarito comentado

### 1 Classificar compras

Como pensar: Validar → comparar limites em ordem → retornar texto.

In [7]:
def classificar_compra(total):
    if total < 0:
        raise ValueError("Total não pode ser negativo.")
    if total < 100:
        return "Pequena"
    if total < 300:
        return "Média"
    return "Grande"

assert classificar_compra(99) == "Pequena"
assert classificar_compra(100) == "Média"
assert classificar_compra(300) == "Grande"

### 2 Reutilizar em lista e tabela

Como pensar: Criar lista → chamar função por valor → apply reutiliza a regra.

In [8]:
classes = []
for valor in [90,300,19]:
    classes.append(classificar_compra(valor))
print(classes)
vendas["classe"] = vendas["total"].apply(classificar_compra)
display(vendas)

['Pequena', 'Grande', 'Pequena']


,produto,preco,quantidade,desconto,total,classe
0,A,50,2,10,90.0,Pequena
1,B,100,3,0,300.0,Grande
2,C,20,1,5,19.0,Pequena


### 3 Compor funções

Como pensar: Calcular → classificar → retornar dicionário; não duplicar regras.

In [9]:
def resumir_compra(preco, quantidade, desconto=0):
    total = total_compra(preco, quantidade, desconto)
    return {"total":total,"classe":classificar_compra(total)}

print(resumir_compra(50,2,10))
try:
    resumir_compra(50,2,120)
except ValueError as erro:
    print("Erro esperado:",erro)
else:
    raise AssertionError("Esperava rejeição de desconto inválido.")

{'total': 90.0, 'classe': 'Pequena'}
Erro esperado: Desconto precisa estar entre 0 e 100.


# Oficina de funções e reutilização

Definir prepara uma função; chamar executa seu corpo. Em notebooks, execute primeiro a célula com def. Faça os exercícios antes de consultar as soluções. As oficinas usam dados próprios e podem ser executadas após os imports.

## Demo Definir chamar e guardar
Em saudacao("Ana"), "Ana" entra no parâmetro nome. return devolve o texto; a variável mensagem recebe esse texto. Sem parênteses, saudacao é o objeto função, não seu resultado. Uma função sem return explícito devolve None.

In [10]:
def saudacao(nome):
    return f"Olá, {nome}!"

mensagem = saudacao("Ana")
print(mensagem)
print(callable(saudacao))

def mostrar(nome):
    print(f"Olá, {nome}!")

retorno = mostrar("Ana")
print("Retorno de mostrar:", retorno)

Olá, Ana!
True
Olá, Ana!
Retorno de mostrar: None


## Demo Argumentos e chamada indireta
Posicionais seguem a ordem; nomeados usam o nome do parâmetro. apply recebe a função, sem executá-la antecipadamente. apply(dobrar) chama dobrar para cada valor. apply(dobrar()) seria uma chamada imediata sem o argumento obrigatório.

In [11]:
def dobrar(valor):
    return valor * 2

print(dobrar(3))
print(dobrar(valor=3))
serie = pd.Series([1,2,3])
print(serie.apply(dobrar).tolist())

6
6
[2, 4, 6]


## Demo Lambda
lambda x: x * 2 cria uma função curta cujo resultado é a expressão. apply chama essa função com cada elemento. x é um nome local, não uma coluna. Para esta multiplicação, serie * 2 é uma alternativa vetorizada mais direta. Lambda também é útil como key de sorted para indicar o critério de ordenação.

In [12]:
print(serie.apply(lambda x: x * 2).tolist())
print((serie * 2).tolist())
compras = [{"produto":"A","total":90},{"produto":"B","total":30}]
print(sorted(compras, key=lambda compra: compra["total"]))

[2, 4, 6]
[2, 4, 6]
[{'produto': 'B', 'total': 30}, {'produto': 'A', 'total': 90}]


## Pratique a transformação
Para cada atividade, escreva entrada → processamento → saída. Compare o código antes e depois: onde a regra fica definida? Quantos lugares precisam mudar quando ela muda? Funções não tornam todo código mais rápido; o ganho aqui é organização, reutilização e manutenção.

### 1 Faça a chamada

A função já existe. Chame com Ana e Bruno; guarde os dois resultados e depois mostre-os.

In [13]:
def boas_vindas(nome):
    return f"Bem-vindo, {nome}!"
# Sua solução aqui.

### 2 Aqui repete crie uma função

O cálculo de desconto se repete. Crie aplicar_desconto(preco, percentual) e reproduza os três resultados.

In [14]:
a = 100 * (1 - 10/100)
b = 200 * (1 - 10/100)
c = 80 * (1 - 10/100)
# Sua solução aqui.

### 3 Ajuste a regra uma vez

Crie a versão validada: preço não negativo; percentual entre 0 e 100. Teste -1 e 110 e capture ValueError.

In [15]:
# Parta do exercício anterior.
# Sua solução aqui.

### 4 Print não devolve o total

Corrija a função para guardar o resultado e somar frete de 15.

In [16]:
def calcular_total(preco, quantidade):
    print(preco * quantidade)
# Sua solução aqui.

### 5 Chamadas com parâmetros nomeados

Crie descrever(nome, cidade="Curitiba"). Chame com cidade padrão e depois com cidade="Cuiabá".

In [17]:

# Sua solução aqui.

### 6 Lista com for

Use aplicar_desconto para [50,100,200], desconto 20, e guarde os resultados em uma lista.

In [18]:
precos = [50,100,200]
# Sua solução aqui.

### 7 Função como argumento

Crie classificar(valor): menor que 100 Pequena, demais Grande. Use apply(classificar) em uma Series. Não use apply(classificar()).

In [19]:
valores = pd.Series([50,100,150])
# Sua solução aqui.

### 8 A mesma regra com lambda

Reproduza a classificação anterior com lambda e confira igualdade. Explique qual opção usaria se a regra precisasse de validações.

In [20]:

# Sua solução aqui.

### 9 Ordenar por um campo

Ordene os produtos por preço crescente usando sorted e key com lambda. A lista original deve continuar igual.

In [21]:
produtos = [{"nome":"A","preco":30},{"nome":"B","preco":10},{"nome":"C","preco":20}]
# Sua solução aqui.

### 10 Compose a solução

Crie resumo(preco, quantidade, desconto=10) que reutilize aplicar_desconto e classificar e retorne subtotal, total e classe.

In [22]:

# Sua solução aqui.

### 11 Encontre os erros de chamada

Diagnostique: classificar(), classificar(valor=50, extra=1), Classificar(50). Capture os erros e escreva chamadas corretas.

In [23]:

# Sua solução aqui.

### 12 Escolha def lambda ou operação direta

Dobre uma coluna com operação vetorizada; ordene registros com lambda; justifique def para desconto validado.

In [24]:
numeros = pd.Series([2,4,6])
# Sua solução aqui.

## Soluções da oficina

### 1 Faça a chamada

A definição sozinha não mostra a mensagem. Cada chamada fornece um argumento e recebe um resultado.

In [25]:
ana = boas_vindas("Ana")
bruno = boas_vindas(nome="Bruno")
print(ana, bruno)
assert ana == "Bem-vindo, Ana!"

Bem-vindo, Ana! Bem-vindo, Bruno!


### 2 Aqui repete crie uma função

A regra passa a ter um lugar único; preço e percentual variam por chamada.

In [26]:
def aplicar_desconto(preco, percentual):
    return preco * (1-percentual/100)

assert [aplicar_desconto(p,10) for p in [100,200,80]] == [90,180,72]
print(aplicar_desconto(100,15))

85.0


### 3 Ajuste a regra uma vez

A validação protege todas as chamadas, sem copiar o if para cada compra.

In [27]:
def aplicar_desconto(preco, percentual=10):
    if preco < 0 or not 0 <= percentual <= 100:
        raise ValueError("Preço ou desconto inválido.")
    return preco * (1-percentual/100)

for preco, desconto in [(-1,10),(100,110)]:
    try:
        aplicar_desconto(preco,desconto)
    except ValueError as erro:
        print(erro)
    else:
        raise AssertionError("Entrada inválida deveria ser rejeitada.")
assert aplicar_desconto(100) == 90

Preço ou desconto inválido.
Preço ou desconto inválido.


### 4 Print não devolve o total

return permite usar o resultado em outra conta. print pode ser usado fora da função.

In [28]:
def calcular_total(preco, quantidade):
    return preco * quantidade

subtotal = calcular_total(20,3)
final = subtotal + 15
print(final)
assert final == 75

75


### 5 Chamadas com parâmetros nomeados

Argumentos nomeados tornam a chamada explícita; o padrão é usado quando cidade é omitida.

In [29]:
def descrever(nome, cidade="Curitiba"):
    return f"{nome} mora em {cidade}."

print(descrever("Ana"))
print(descrever(cidade="Cuiabá", nome="Bruno"))
assert descrever("Ana") == "Ana mora em Curitiba."

Ana mora em Curitiba.
Bruno mora em Cuiabá.


### 6 Lista com for

O for repete a chamada; a função define o que fazer em cada chamada.

In [30]:
com_desconto = []
for preco in precos:
    com_desconto.append(aplicar_desconto(preco,20))
print(com_desconto)
assert com_desconto == [40,80,160]

[40.0, 80.0, 160.0]


### 7 Função como argumento

Passar classificar entrega a função ao Pandas, que fornece os valores ao chamá-la.

In [31]:
def classificar(valor):
    return "Pequena" if valor < 100 else "Grande"

classes = valores.apply(classificar)
print(classes.tolist())
assert classes.tolist() == ["Pequena","Grande","Grande"]

['Pequena', 'Grande', 'Grande']


### 8 A mesma regra com lambda

Uma expressão curta cabe em lambda; com validações e várias etapas, def costuma ser mais legível.

In [32]:
classes_lambda = valores.apply(lambda valor: "Pequena" if valor < 100 else "Grande")
assert classes_lambda.equals(classes)
print(classes_lambda.tolist())

['Pequena', 'Grande', 'Grande']


### 9 Ordenar por um campo

key recebe uma função que extrai o critério; sorted devolve uma nova lista.

In [33]:
ordenados = sorted(produtos, key=lambda produto: produto["preco"])
print(ordenados)
assert [p["nome"] for p in ordenados] == ["B","C","A"]
assert produtos[0]["nome"] == "A"

[{'nome': 'B', 'preco': 10}, {'nome': 'C', 'preco': 20}, {'nome': 'A', 'preco': 30}]


### 10 Compose a solução

Dividir responsabilidades permite reutilizar regras sem reescrevê-las.

In [34]:
def resumo(preco, quantidade, desconto=10):
    subtotal = preco * quantidade
    total = aplicar_desconto(subtotal,desconto)
    return {"subtotal":subtotal,"total":total,"classe":classificar(total)}

print(resumo(50,3))
assert resumo(50,3)["total"] == 135

{'subtotal': 150, 'total': 135.0, 'classe': 'Grande'}


### 11 Encontre os erros de chamada

Faltou argumento; há um argumento desconhecido; Python distingue maiúsculas e minúsculas. Se aparecer NameError por falta de definição, execute a célula do def primeiro.

In [35]:
for chamada in [lambda: classificar(), lambda: classificar(valor=50,extra=1), lambda: Classificar(50)]:
    try:
        chamada()
    except (TypeError,NameError) as erro:
        print(type(erro).__name__,str(erro))
print(classificar(50))

TypeError classificar() missing 1 required positional argument: 'valor'
TypeError classificar() got an unexpected keyword argument 'extra'
NameError name 'Classificar' is not defined
Pequena


### 12 Escolha def lambda ou operação direta

Use a forma mais clara para a tarefa: operação entre colunas, critério curto de ordenação ou regra nomeada com validação.

In [36]:
dobrados = numeros * 2
assert dobrados.tolist() == [4,8,12]
print(dobrados.tolist())
print(sorted(produtos,key=lambda p:p["preco"]))

[4, 8, 12]
[{'nome': 'B', 'preco': 10}, {'nome': 'C', 'preco': 20}, {'nome': 'A', 'preco': 30}]


Referência: https://docs.python.org/3/tutorial/controlflow.html#defining-functions . lambda contém uma expressão; use def quando precisar de várias etapas, validação ou uma regra reutilizada com nome.